# Satellite Coordination Pressure Model

This notebook trains a simulation-support model for the **Satellite Autonomy Interoperability Layer (SAIL)** project.

The current satellite catalog does not include real labels for collisions, conjunction negotiations, or autonomous maneuver outcomes. So this notebook creates a transparent surrogate target called `coordination_pressure_tier`. The target is meant to seed simulations, not to make operational safety decisions.

**Best current model choice:** `HistGradientBoostingClassifier`.

Why this model:

- the dataset is tabular and mixes orbital, mission, and operator fields;
- coordination pressure is nonlinear, especially around altitude bands and orbit regimes;
- gradient-boosted trees work well on medium-sized tabular datasets;
- the model is stronger than a random forest baseline in cross-validation while still being easy to retrain locally.

Safety note: this is a research model for simulation. It is not flight software, a certified conjunction assessment product, or collision avoidance advice.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 80)
RANDOM_STATE = 42
CURRENT_YEAR = 2026

## 1. Load the satellite catalog

For local development in the original paper directory, the catalog is available at `../database.csv`. For a fresh clone, place the catalog at `data/database.csv`.

In [ ]:
def find_data_path() -> Path:
    candidates = [
        Path("data/database.csv"),
        Path("../database.csv"),
        Path("database.csv"),
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError("Place database.csv in data/database.csv or one directory above the repo.")

DATA_PATH = find_data_path()
df_raw = pd.read_csv(DATA_PATH)
print(f"Loaded {DATA_PATH} with shape {df_raw.shape}")
df_raw.head(3)

In [ ]:
df_raw.info()

## 2. Clean fields and engineer simulation features

The catalog has a mix of numeric columns, dates, and categorical mission descriptors. We clean numeric values, parse launch dates, and add features that matter for simulation:

- mean altitude;
- altitude span;
- satellite age;
- altitude-band density inside the catalog;
- operator concentration.

In [ ]:
NUMERIC_COLUMNS = [
    "Perigee (Kilometers)",
    "Apogee (Kilometers)",
    "Eccentricity",
    "Inclination (Degrees)",
    "Period (Minutes)",
    "Launch Mass (Kilograms)",
    "Expected Lifetime (Years)",
]


def clean_number(series: pd.Series) -> pd.Series:
    text = series.astype(str).str.replace(",", "", regex=False)
    numeric_text = text.str.extract(r"([-+]?\d*\.?\d+)")[0]
    return pd.to_numeric(numeric_text, errors="coerce").replace([np.inf, -np.inf], np.nan)


def prepare_catalog(df: pd.DataFrame) -> pd.DataFrame:
    work = df.copy()
    work["Class of Orbit"] = work["Class of Orbit"].astype(str).str.strip().replace({"nan": np.nan})

    for column in NUMERIC_COLUMNS:
        work[column] = clean_number(work[column])

    work["Date of Launch"] = pd.to_datetime(work["Date of Launch"], errors="coerce")
    work["launch_year"] = work["Date of Launch"].dt.year
    work["satellite_age_years"] = CURRENT_YEAR - work["launch_year"]
    work["mean_altitude_km"] = work[["Perigee (Kilometers)", "Apogee (Kilometers)"]].mean(axis=1)
    work["altitude_span_km"] = work["Apogee (Kilometers)"] - work["Perigee (Kilometers)"]

    altitude_bins = pd.cut(
        work["mean_altitude_km"].clip(lower=0, upper=40000),
        bins=np.arange(0, 40001, 250),
        include_lowest=True,
    )
    density = altitude_bins.map(altitude_bins.value_counts(normalize=True)).astype(float).fillna(0)
    if density.max() > density.min():
        work["altitude_band_density"] = (density - density.min()) / (density.max() - density.min())
    else:
        work["altitude_band_density"] = density

    operator_share = work["Operator/Owner"].map(work["Operator/Owner"].value_counts(normalize=True)).fillna(0)
    if operator_share.max() > operator_share.min():
        work["operator_concentration"] = (operator_share - operator_share.min()) / (operator_share.max() - operator_share.min())
    else:
        work["operator_concentration"] = operator_share

    return work

catalog = prepare_catalog(df_raw)
catalog[["Official Name of Satellite", "Class of Orbit", "Purpose", "mean_altitude_km", "altitude_band_density"]].head()

## 3. Create the simulation target

`coordination_pressure_score` is a transparent heuristic designed for simulation seeding. It is higher for satellites that are more likely to create coordination pressure in the paper's three-layer framework:

- LEO objects, especially dense low-altitude bands;
- communications and navigation missions;
- higher eccentricity or wider altitude span;
- higher operator concentration;
- missing expected lifetime metadata.

The score is converted into three balanced tiers: `low`, `medium`, and `high`.

In [ ]:
def add_coordination_pressure_target(df: pd.DataFrame) -> pd.DataFrame:
    work = df.copy()
    purpose = work["Purpose"].fillna("").str.lower()
    orbit = work["Class of Orbit"].fillna("").str.upper()

    score = (
        0.05
        + orbit.eq("LEO") * 0.30
        + orbit.eq("MEO") * 0.15
        + orbit.eq("ELLIPTICAL") * 0.20
        + purpose.str.contains("communications", regex=False) * 0.18
        + purpose.str.contains("navigation", regex=False) * 0.12
        + purpose.str.contains("earth observation", regex=False) * 0.10
        + ((work["mean_altitude_km"] >= 300) & (work["mean_altitude_km"] <= 1400)) * 0.18
        + (work["Eccentricity"].fillna(0) > 0.05) * 0.08
        + work["altitude_band_density"].fillna(0) * 0.18
        + work["operator_concentration"].fillna(0) * 0.10
        + work["Expected Lifetime (Years)"].isna() * 0.04
    )

    work["coordination_pressure_score"] = np.clip(score.astype(float), 0, 1)
    work["coordination_pressure_tier"] = pd.qcut(
        work["coordination_pressure_score"],
        q=3,
        labels=["low", "medium", "high"],
        duplicates="drop",
    ).astype(str)
    return work

model_df = add_coordination_pressure_target(catalog)
model_df["coordination_pressure_tier"].value_counts().sort_index()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
model_df["coordination_pressure_score"].hist(bins=30, ax=ax)
ax.set_title("Coordination pressure score distribution")
ax.set_xlabel("Score")
ax.set_ylabel("Satellites")
plt.show()

## 4. Train/test split and preprocessing

We use only fields that could plausibly be available before a simulation run. The preprocessing pipeline imputes missing numeric/categorical values, scales numeric fields, and one-hot encodes categorical fields.

In [ ]:
FEATURES = [
    "Users",
    "Purpose",
    "Class of Orbit",
    "Type of Orbit",
    "Country of Operator/Owner",
    "Perigee (Kilometers)",
    "Apogee (Kilometers)",
    "Eccentricity",
    "Inclination (Degrees)",
    "Period (Minutes)",
    "Launch Mass (Kilograms)",
    "Expected Lifetime (Years)",
    "launch_year",
    "satellite_age_years",
    "mean_altitude_km",
    "altitude_span_km",
]

X = model_df[FEATURES].replace([np.inf, -np.inf], np.nan)
y = model_df["coordination_pressure_tier"]

categorical_features = [column for column in FEATURES if X[column].dtype == "object"]
numeric_features = [column for column in FEATURES if column not in categorical_features]

preprocess = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            numeric_features,
        ),
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False, min_frequency=5)),
                ]
            ),
            categorical_features,
        ),
    ],
    sparse_threshold=0,
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

print(X_train.shape, X_test.shape)

## 5. Compare candidate models

The candidates are:

- `DummyClassifier`: sanity-check baseline;
- `RandomForestClassifier`: strong, stable tree baseline;
- `HistGradientBoostingClassifier`: selected model for this tabular simulation task.

The notebook chooses the model with the highest mean weighted F1 score under stratified cross-validation.

In [ ]:
models = {
    "dummy_most_frequent": DummyClassifier(strategy="most_frequent"),
    "random_forest": RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE,
        class_weight="balanced",
        min_samples_leaf=2,
    ),
    "hist_gradient_boosting": HistGradientBoostingClassifier(
        random_state=RANDOM_STATE,
        learning_rate=0.06,
        max_iter=200,
        l2_regularization=0.05,
    ),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results = []

for name, model in models.items():
    pipeline = Pipeline(steps=[("preprocess", preprocess), ("model", model)])
    scores = cross_validate(
        pipeline,
        X,
        y,
        cv=cv,
        scoring=["accuracy", "f1_weighted"],
        n_jobs=-1,
    )
    results.append(
        {
            "model": name,
            "accuracy_mean": scores["test_accuracy"].mean(),
            "f1_weighted_mean": scores["test_f1_weighted"].mean(),
            "f1_weighted_std": scores["test_f1_weighted"].std(),
        }
    )

results_df = pd.DataFrame(results).sort_values("f1_weighted_mean", ascending=False)
results_df

In [ ]:
best_model_name = results_df.iloc[0]["model"]
best_model = models[best_model_name]

best_pipeline = Pipeline(steps=[("preprocess", preprocess), ("model", best_model)])
best_pipeline.fit(X_train, y_train)

y_pred = best_pipeline.predict(X_test)

print(f"Selected model: {best_model_name}")
print(f"Holdout accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(f"Holdout weighted F1: {f1_score(y_test, y_pred, average='weighted'):.3f}")
print(classification_report(y_test, y_pred))

In [ ]:
labels = sorted(y.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(labels)), labels=labels)
ax.set_yticks(range(len(labels)), labels=labels)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Holdout confusion matrix")
for row in range(len(labels)):
    for col in range(len(labels)):
        ax.text(col, row, cm[row, col], ha="center", va="center", color="black")
fig.colorbar(im, ax=ax)
plt.show()

## 6. Save model artifacts and simulation seeds

The trained model can seed future simulations with data-derived coordination pressure tiers. Binary model artifacts are intentionally ignored by Git; regenerate them by rerunning the notebook.

In [ ]:
MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)

model_path = MODELS_DIR / "satellite_coordination_pressure_model.joblib"
metrics_path = MODELS_DIR / "satellite_coordination_pressure_metrics.json"
predictions_path = MODELS_DIR / "satellite_coordination_pressure_predictions.csv"

joblib.dump(best_pipeline, model_path)

metrics = {
    "selected_model": best_model_name,
    "holdout_accuracy": float(accuracy_score(y_test, y_pred)),
    "holdout_f1_weighted": float(f1_score(y_test, y_pred, average="weighted")),
    "cv_results": results_df.to_dict(orient="records"),
    "target": "coordination_pressure_tier",
    "target_note": "Surrogate simulation label derived from transparent orbital and mission heuristics.",
}
metrics_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")

simulation_seeds = model_df[[
    "Official Name of Satellite",
    "NORAD Number",
    "Operator/Owner",
    "Purpose",
    "Class of Orbit",
    "mean_altitude_km",
    "coordination_pressure_score",
    "coordination_pressure_tier",
]].copy()
simulation_seeds["predicted_coordination_pressure_tier"] = best_pipeline.predict(X)
simulation_seeds.to_csv(predictions_path, index=False)

print(f"Saved model to {model_path}")
print(f"Saved metrics to {metrics_path}")
print(f"Saved simulation seeds to {predictions_path}")

## 7. How to use this in the simulator

Near-term integration idea:

1. Use `satellite_coordination_pressure_predictions.csv` to assign each simulated satellite a starting risk tier.
2. Convert `high` tier objects into more conservative maneuver thresholds and shorter message expiration windows.
3. Use `medium` tier objects for nominal SAIL coordination tests.
4. Use `low` tier objects as background traffic.

This connects the CSV data to the SAIL architecture without pretending the model is an operational collision predictor.